In [1]:
import subprocess
subprocess.run(["pip", "install", "pyarrow", "tqdm", "psutil", "-q"])

import pandas as pd
import numpy as np
import json, gzip, gc
import matplotlib.pyplot as plt
import psutil
from pathlib import Path
from tqdm import tqdm
from collections import Counter
import pyarrow as pa
import pyarrow.parquet as pq

def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"
print(f"💾 {ram_usage()}")

💾 RAM: 6.0/7.9 GB (76%)


## Cell 2.2 — Cấu hình đường dẫn & tham số

In [2]:
ROOT_DIR      = Path().resolve().parent
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
TEMP_DIR      = ROOT_DIR / "data" / "processed" / "_temp_chunks"

for d in [PROCESSED_DIR, SAMPLE_DIR, FIGURES_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# CHUNKING CONFIG
# CHUNK_SIZE = 100_000
CHUNK_SIZE           = 100_000
MIN_TEXT_LENGTH      = 10
MIN_REVIEWS_PER_USER = 5
MIN_REVIEWS_PER_ITEM = 5
RANDOM_SEED          = 42
np.random.seed(RANDOM_SEED)

print(f"   Chunk size : {CHUNK_SIZE:,} dòng/lần")
print(f"   Review file: {REVIEW_PATH.stat().st_size/(1024**3):.2f} GB")
print(f"   Meta file  : {META_PATH.stat().st_size/(1024**3):.2f} GB")
print(f"{ram_usage()}")

   Chunk size : 100,000 dòng/lần
   Review file: 6.62 GB
   Meta file  : 3.77 GB
RAM: 5.9/7.9 GB (75%)


In [3]:
#dem so dong cua review va meta
def count_lines(filepath):
    count = 0
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for _ in tqdm(f, desc=filepath.name[:35], unit=" lines"):
            count += 1
    return count

n_reviews = count_lines(REVIEW_PATH)
n_meta    = count_lines(META_PATH)
n_chunks  = (n_reviews // CHUNK_SIZE) + 1

print(f"   Review: {n_reviews:,} dòng → {n_chunks} chunks")
print(f"   Meta  : {n_meta:,} dòng")

Clothing_Shoes_and_Jewelry.jsonl.gz: 0 lines [00:00, ? lines/s]

Clothing_Shoes_and_Jewelry.jsonl.gz: 66033346 lines [03:43, 294908.17 lines/s]
meta_Clothing_Shoes_and_Jewelry.jso: 7218481 lines [01:56, 62089.31 lines/s]

   Review: 66,033,346 dòng → 661 chunks
   Meta  : 7,218,481 dòng


 Hàm làm sạch 1 chunk review

In [4]:
def clean_review_chunk(df):
    if df.empty:
        return None

    # cột cần làm sạch
    needed = ['rating', 'text', 'title', 'user_id',
              'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
    df = df[[c for c in needed if c in df.columns]].copy()

    # drop thiếu rating hoặc text
    df = df.dropna(subset=['rating', 'text'])
    if df.empty: return None

    # Chuẩn hóa kiểu dữ liệu
    df['rating']       = pd.to_numeric(df['rating'], errors='coerce')
    df['helpful_vote'] = pd.to_numeric(df.get('helpful_vote', 0), errors='coerce').fillna(0).astype(int)
    df = df.dropna(subset=['rating'])
    if df.empty: return None
    df['rating'] = df['rating'].astype('float32')

    # Timestamp → year, month
    if 'timestamp' in df.columns:
        dt = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
        df['year']  = dt.dt.year.astype('Int16')
        df['month'] = dt.dt.month.astype('Int8')

    # Tạo nhãn Sentiment
    df['sentiment'] = df['rating'].map(
        lambda r: 'positive' if r >= 4 else ('neutral' if r == 3 else 'negative')
    ).astype('category')

    # Lọc review quá ngắn (< 10 ký tự = noise)
    df['text_length'] = df['text'].astype(str).str.len().astype('int32')
    df = df[df['text_length'] >= MIN_TEXT_LENGTH]
    if df.empty: return None

    # Loại duplicate trong cùng chunk
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')

    return df.reset_index(drop=True)

 Xử lý toàn bộ Review theo chunking

In [5]:
# Xóa chunks cũ nếu chạy lại
for f in TEMP_DIR.glob("chunk_*.parquet"):
    f.unlink()

chunk_id      = 0
total_raw     = 0
total_clean   = 0
current_chunk = []

print(f"   Chunk size: {CHUNK_SIZE:,} | ~{n_chunks} chunks")
print()

with gzip.open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    pbar = tqdm(f, total=n_reviews, desc="Processing", unit=" lines")

    for line in pbar:
        total_raw += 1
        try:
            current_chunk.append(json.loads(line.strip()))
        except:
            continue

        if len(current_chunk) >= CHUNK_SIZE:
            df_chunk       = pd.DataFrame(current_chunk)
            df_chunk_clean = clean_review_chunk(df_chunk)

            if df_chunk_clean is not None:
                chunk_path = TEMP_DIR / f"chunk_{chunk_id:04d}.parquet"
                df_chunk_clean.to_parquet(chunk_path, index=False)
                total_clean += len(df_chunk_clean)

            chunk_id      += 1
            current_chunk  = []
            del df_chunk, df_chunk_clean
            gc.collect()  # Giải phóng RAM ngay lập tức

            pbar.set_postfix({
                'chunks': chunk_id,
                'clean' : f"{total_clean:,}",
                'RAM'   : f"{psutil.virtual_memory().percent:.0f}%"
            })

    # Xử lý phần dư cuối file
    if current_chunk:
        df_chunk       = pd.DataFrame(current_chunk)
        df_chunk_clean = clean_review_chunk(df_chunk)
        if df_chunk_clean is not None:
            (TEMP_DIR / f"chunk_{chunk_id:04d}.parquet").parent
            df_chunk_clean.to_parquet(TEMP_DIR / f"chunk_{chunk_id:04d}.parquet", index=False)
            total_clean += len(df_chunk_clean)
        chunk_id += 1

gc.collect()
print(f"   Tổng đọc  : {total_raw:,}")
print(f"   Sau sạch  : {total_clean:,} ({total_clean/total_raw*100:.1f}% giữ lại)")
print(f"   Số chunks : {chunk_id}")
print(f"{ram_usage()}")

   Chunk size: 100,000 | ~661 chunks



Processing: 100%|██████████| 66033346/66033346 [23:11<00:00, 47454.90 lines/s, chunks=660, clean=62,350,931, RAM=80%]  


   Tổng đọc  : 66,033,346
   Sau sạch  : 62,381,837 (94.5% giữ lại)
   Số chunks : 661
RAM: 6.3/7.9 GB (80%)


 Ghép tất cả chunks thành 1 file parquet

In [6]:
chunk_files = sorted(TEMP_DIR.glob("chunk_*.parquet"))
output_path = PROCESSED_DIR / "review_clean.parquet"
writer = None
for chunk_file in tqdm(chunk_files, desc="Merging chunks"):
    table = pq.read_table(chunk_file)
    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema, compression='snappy')
    writer.write_table(table)
    del table
    gc.collect()

if writer:
    writer.close()

# Xóa thư mục chunks tạm
import shutil
shutil.rmtree(TEMP_DIR)
sz = output_path.stat().st_size / (1024**3)
print(f"\n review_clean.parquet: {sz:.2f} GB")
print(f"{ram_usage()}")

Merging chunks:   0%|          | 0/661 [00:00<?, ?it/s]

Merging chunks: 100%|██████████| 661/661 [03:47<00:00,  2.90it/s]



 review_clean.parquet: 8.29 GB
RAM: 6.2/7.9 GB (78%)


 Xử lý Meta dataset

In [8]:
def load_and_clean_meta_chunked(filepath, chunk_size=50_000):
    TEMP_META_DIR = PROCESSED_DIR / "_temp_meta_chunks"
    TEMP_META_DIR.mkdir(exist_ok=True)

    # Xóa chunks cũ nếu có
    for f in TEMP_META_DIR.glob("*.parquet"):
        f.unlink()

    chunk_id      = 0
    total_loaded  = 0
    current_chunk = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading meta", unit=" lines"):
            try:
                current_chunk.append(json.loads(line.strip()))
            except:
                continue

            if len(current_chunk) >= chunk_size:
                df = pd.DataFrame(current_chunk)
                df = _clean_meta_df(df)          
                if df is not None:
                    df.to_parquet(TEMP_META_DIR / f"meta_chunk_{chunk_id:04d}.parquet", index=False)
                    total_loaded += len(df)
                chunk_id += 1
                current_chunk = []
                del df
                gc.collect()

    # Xử lý phần dư
    if current_chunk:
        df = pd.DataFrame(current_chunk)
        df = _clean_meta_df(df)
        if df is not None:
            df.to_parquet(TEMP_META_DIR / f"meta_chunk_{chunk_id:04d}.parquet", index=False)
            total_loaded += len(df)
        del df
        gc.collect()

    # Ghép chunks lại
    chunk_files = sorted(TEMP_META_DIR.glob("*.parquet"))
    writer = None
    output = PROCESSED_DIR / "meta_clean.parquet"

    for cf in tqdm(chunk_files, desc="Merging"):
        table = pq.read_table(cf)
        if writer is None:
            writer = pq.ParquetWriter(output, table.schema, compression='snappy')
        writer.write_table(table)
        del table
        gc.collect()

    if writer:
        writer.close()

    # Xóa temp
    import shutil
    shutil.rmtree(TEMP_META_DIR)

    sz = output.stat().st_size / (1024**2)
    print(f" {ram_usage()}")


def _clean_meta_df(df):
    """Làm sạch 1 chunk meta"""
    if df is None or df.empty:
        return None

    needed = ['parent_asin','title','price','description',
              'categories','average_rating','rating_number',
              'store','main_category']
    df = df[[c for c in needed if c in df.columns]].copy()

    df = df.dropna(subset=['parent_asin','title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')

    if 'price' in df.columns:
        df['price'] = pd.to_numeric(
            df['price'].astype(str).str.replace(r'[^\d.]','',regex=True),
            errors='coerce'
        ).astype('float32')

    if 'description' in df.columns:
        df['description'] = df['description'].apply(
            lambda x: ' '.join(x) if isinstance(x, list) else (str(x) if pd.notna(x) else '')
        )

    if 'categories' in df.columns:
        def extract_cat(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                return inner[0] if len(inner) > 0 else 'Unknown'
            return 'Unknown'
        df['main_category'] = df['categories'].apply(extract_cat)

    return df.reset_index(drop=True)

load_and_clean_meta_chunked(META_PATH, chunk_size=50_000)

Reading meta: 7218481 lines [08:25, 14272.32 lines/s]
Merging: 100%|██████████| 145/145 [00:56<00:00,  2.59it/s]

 RAM: 6.6/7.9 GB (84%)


 Merge Review + Meta (chunked)

In [9]:
meta_for_merge = pd.read_parquet(
    PROCESSED_DIR / "meta_clean.parquet",
    columns=[c for c in ['parent_asin','title','price','main_category']
             if (PROCESSED_DIR / "meta_clean.parquet").exists()]
)
gc.collect()
print(f"Loaded meta: {meta_for_merge.shape} | {ram_usage()}")

review_pf     = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
merged_output = PROCESSED_DIR / "merged_clean.parquet"
writer_m      = None
total_merged  = 0
n_rg          = review_pf.metadata.num_row_groups

for i in tqdm(range(n_rg), desc="Merging"):
    chunk  = review_pf.read_row_group(i).to_pandas()
    merged = chunk.merge(meta_for_merge, on='parent_asin', how='left')

    if 'title_x' in merged.columns:
        merged = merged.rename(columns={'title_x':'review_title','title_y':'product_title'})

    table = pa.Table.from_pandas(merged, preserve_index=False)
    if writer_m is None:
        writer_m = pq.ParquetWriter(merged_output, table.schema, compression='snappy')
    writer_m.write_table(table)
    total_merged += len(merged)

    del chunk, merged, table
    gc.collect()

if writer_m:
    writer_m.close()

sz = merged_output.stat().st_size / (1024**3)
print(f"\n merged_clean.parquet: {total_merged:,} rows | {sz:.2f} GB")
print(f" {ram_usage()}")

Loaded meta: (7218481, 4) | RAM: 6.5/7.9 GB (82%)


Merging: 100%|██████████| 661/661 [2:24:35<00:00, 13.12s/it]      


 merged_clean.parquet: 62,381,837 rows | 11.09 GB
 RAM: 6.2/7.9 GB (79%)


 Cold-start filter cho Recommendation System

In [10]:
# Lọc user/item có ít hơn 5 reviews (cold-start problem)
# Dùng 2 pass:
#   Pass 1: đọc hết file, đếm user/item
#   Pass 2: đọc lại, chỉ giữ user/item hợp lệ

review_pf = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
n_rg      = review_pf.metadata.num_row_groups

# Pass 1: Đếm user/item counts
print("Pass 1/2: Đếm user/item counts...")
user_counter = Counter()
item_counter = Counter()

for i in tqdm(range(n_rg), desc="Counting"):
    chunk = review_pf.read_row_group(i, columns=['user_id','parent_asin']).to_pandas()
    user_counter.update(chunk['user_id'].tolist())
    item_counter.update(chunk['parent_asin'].tolist())
    del chunk; gc.collect()

valid_users = {u for u,c in user_counter.items() if c >= MIN_REVIEWS_PER_USER}
valid_items = {p for p,c in item_counter.items() if c >= MIN_REVIEWS_PER_ITEM}
print(f"   Valid users: {len(valid_users):,} / {len(user_counter):,}")
print(f"   Valid items: {len(valid_items):,} / {len(item_counter):,}")

#  Pass 2: Lọc & Lưu
print("\nPass 2/2: Lọc và lưu...")
rec_output = PROCESSED_DIR / "review_for_rec.parquet"
writer_r   = None
total_rec  = 0

for i in tqdm(range(n_rg), desc="Filtering"):
    chunk    = review_pf.read_row_group(i).to_pandas()
    filtered = chunk[
        chunk['user_id'].isin(valid_users) &
        chunk['parent_asin'].isin(valid_items)
    ]
    if len(filtered) > 0:
        table = pa.Table.from_pandas(filtered, preserve_index=False)
        if writer_r is None:
            writer_r = pq.ParquetWriter(rec_output, table.schema, compression='snappy')
        writer_r.write_table(table)
        total_rec += len(filtered)
        del table
    del chunk, filtered; gc.collect()

if writer_r: writer_r.close()

sz = rec_output.stat().st_size / (1024**2)
print(f"\n review_for_rec.parquet: {total_rec:,} rows | {sz:.1f} MB")
print(f" {ram_usage()}")

Pass 1/2: Đếm user/item counts...


Counting: 100%|██████████| 661/661 [02:53<00:00,  3.81it/s]


   Valid users: 3,197,295 / 21,942,115
   Valid items: 1,274,462 / 7,039,165

Pass 2/2: Lọc và lưu...


Filtering: 100%|██████████| 661/661 [1:08:16<00:00,  6.20s/it]


 review_for_rec.parquet: 26,150,403 rows | 3567.3 MB
 RAM: 6.6/7.9 GB (84%)


In [11]:
print(" Tạo file sample 100k rows")
review_pf   = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
sample_rows = []
collected   = 0
target      = 100_000

for i in range(review_pf.metadata.num_row_groups):
    chunk  = review_pf.read_row_group(i).to_pandas()
    n_take = min(len(chunk), target - collected)
    sample_rows.append(chunk.sample(n=n_take, random_state=RANDOM_SEED))
    collected += n_take
    del chunk
    if collected >= target:
        break

df_sample = pd.concat(sample_rows, ignore_index=True)
df_sample.to_parquet(SAMPLE_DIR / "review_sample_100k.parquet", index=False)
print(f" review_sample_100k.parquet: {len(df_sample):,} rows")

 Tạo file sample 100k rows
 review_sample_100k.parquet: 100,000 rows
